# JCP Fall 26 — Week 7: SQL

This notebook goes with the Week 7 SQL slides. No installs: Python already ships with **SQLite**, a real database that lives in one file. We send it SQL as a string and get a pandas DataFrame back.

1. **Predict**: read the cell and write your guess in the `# Guess:` comment *before* you run it.
2. **Fix**: the cell is broken on purpose. Read the error (or the wrong output), find the bug, fix it.
3. **Tinker**: change one value, rerun, and notice what changes.
4. **Your Turn**: replace every `...` with your own SQL.

Examples use a cafe database: **`customers`** and **`orders`**. Your Turn cells use a music streaming app: **`users`**, **`songs`**, and **`plays`**. Same skills, different tables, and there are a lot of Your Turn questions, so pace yourself.

Keep each cell short. If you're stuck for more than 2 minutes, ask your partner, then ask us.

---
## 0. Setup

Run these cells once. The first one builds `week7.db` in the same folder as this notebook (you don't need to read it).

In [ ]:
import sqlite3
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# ---------- cafe database (examples) ----------
customers = pd.DataFrame({
    "customer_id":   [1, 2, 3, 4, 5, 6, 7, 8],
    "customer_name": ["Ava", "Ben", "Chloe", "Dev", "Emma", "Farah", "Gabe", "Hana"],
    "city":          ["Berkeley", "Oakland", "Berkeley", "San Francisco", "Oakland", "Berkeley", "San Francisco", "Berkeley"],
    "nickname":      [None, "Benny", None, "D", None, None, "G", None],
})
menu = {"Latte": 5.25, "Mocha": 5.75, "Matcha": 6.00, "Cold Brew": 4.75, "Tea": 3.50, "Chai Latte": 5.50}
order_rows = []
for order_id in range(1, 41):
    cid = int(rng.choice([1, 1, 2, 3, 3, 3, 4, 5, 6, 6, 7, 9]))      # customer 8 never orders, customer 9 doesn't exist
    drink = str(rng.choice(list(menu)))
    size = rng.choice(["small", "medium", "large", None], p=[0.3, 0.4, 0.2, 0.1])
    price = round(menu[drink] + {"small": -0.5, "medium": 0, "large": 0.75, None: 0}[size], 2)
    day = int(rng.integers(1, 31))
    order_rows.append([order_id, cid, drink, size, price, f"2026-10-{day:02d}"])
orders = pd.DataFrame(order_rows, columns=["order_id", "customer_id", "drink", "size", "price", "order_date"])

# ---------- music streaming database (Your Turn) ----------
users = pd.DataFrame({
    "user_id": range(1, 26),
    "username": ["lofi_luna", "beatsbyben", "kenji", "mango", "vinyl_vic", "zara9", "night_owl", "dj_dani",
                 "ritz", "solar_sam", "pixel_pia", "echo", "moonlit_mo", "tuneup_tia", "rhythm_raj", "aria",
                 "bassline_bo", "cadence", "indie_ines", "loop_leo", "melody_mia", "noah", "groove_gus", "harmony_h", "kai_k"],
    "country": rng.choice(["US", "Canada", "Mexico", "UK", "India"], size=25, p=[0.4, 0.15, 0.15, 0.15, 0.15]),
    "plan": rng.choice(["free", "premium", "student"], size=25, p=[0.4, 0.3, 0.3]),
    "signup_date": [f"2025-{int(m):02d}-{int(d):02d}" for m, d in zip(rng.integers(1, 13, 25), rng.integers(1, 29, 25))],
    "referred_by": [None, None, 1, None, 1, 2, None, 4, 1, None, 5, None, 2, None, 4, 9, None, 4, None, 12,
                    None, 3, None, 9, None],
})
genre_artists = {
    "Pop": ["Neon Harbor", "Sugar Static"], "Indie": ["Paper Kites Club", "Little Lanterns"],
    "Hip-Hop": ["MC Parallax", "Lyric Theory"], "Lo-fi": ["Rainy Window"], "Rock": ["Cinder Avenue"],
    "R&B": ["Velvet Hour", "Juno Rae"],
}
titles = ["Love on Mute", "Glass Skyline", "Midnight Bus", "Paper Planes Again", "Static Love", "Golden Hour Freestyle",
          "Coffee Rings", "Northbound", "Love Letters in Pencil", "Satellite Hearts", "Rooftop Summer", "Slow Fade",
          "Fever Dream", "Late Checkout", "Lovesick Algorithm", "Polaroid", "Second Wind", "Cloud Nine Remix",
          "Backseat Radio", "Low Tide", "Wildfire", "Echo Chamber", "City of Lights", "Daydream Arcade",
          "Bittersweet", "Heatwave", "Snow Globe", "Overtime", "Afterglow", "Window Seat"]
song_rows = []
for i, title in enumerate(titles, start=1):
    genre = list(genre_artists)[i % 6]
    artist = genre_artists[genre][i % len(genre_artists[genre])]
    if genre == "Rock":
        duration = int(rng.integers(220, 360))
    elif genre == "Lo-fi":
        duration = int(rng.integers(95, 180))
    else:
        duration = int(rng.integers(140, 300))
    song_rows.append([i, title, artist, genre, duration, int(rng.integers(2008, 2027))])
songs = pd.DataFrame(song_rows, columns=["song_id", "title", "artist", "genre", "duration_sec", "release_year"])

song_weights = rng.gamma(2.0, 1.0, size=len(songs))
song_weights[[26, 28]] = 0                       # two songs nobody plays
favorite_genre = {"US": "Hip-Hop", "Canada": "Indie", "Mexico": "Pop", "UK": "Rock", "India": "Lo-fi"}
skip_rate = {"Pop": 0.25, "Indie": 0.2, "Hip-Hop": 0.3, "Lo-fi": 0.1, "Rock": 0.35, "R&B": 0.2}
play_rows = []
play_id = 1
for uid in users["user_id"]:
    if uid in (12, 23):                          # two users who signed up but never listened
        continue
    n_plays = int(rng.integers(5, 60))
    country = users.loc[users["user_id"] == uid, "country"].iloc[0]
    w = song_weights * np.where(songs["genre"] == favorite_genre[country], 4.0, 1.0)
    w = w / w.sum()
    for _ in range(n_plays):
        s = songs.iloc[rng.choice(len(songs), p=w)]
        skipped = rng.random() < skip_rate[s["genre"]]
        seconds = int(rng.integers(3, 30)) if skipped else int(s["duration_sec"])
        play_rows.append([play_id, int(uid), int(s["song_id"]), f"2026-{int(rng.integers(8, 11)):02d}-{int(rng.integers(1, 29)):02d}", seconds])
        play_id += 1
plays = pd.DataFrame(play_rows, columns=["play_id", "user_id", "song_id", "play_date", "seconds_listened"])

conn = sqlite3.connect("week7.db")
for name, df in [("customers", customers), ("orders", orders), ("users", users), ("songs", songs), ("plays", plays)]:
    df.to_sql(name, conn, if_exists="replace", index=False)
print("database ready")

This helper is the only Python you need today: it sends a SQL string to the database and returns a DataFrame.

In [ ]:
def sql(query):
    return pd.read_sql(query, conn)

sql("SELECT name FROM sqlite_master WHERE type = 'table'")

In [ ]:
# FULL OUTER JOIN needs SQLite 3.39 or newer. If yours is older, you can skip that one cell later.
sqlite3.sqlite_version

**Example:** peek at each table. The triple quotes `"""` let a query span several lines.

In [ ]:
sql("""
SELECT *
FROM customers
""")

In [ ]:
sql("""
SELECT *
FROM orders
LIMIT 5
""")

**Your Turn:** the music app has **three** tables: `users`, `songs`, and `plays` (one row per time someone played a song). Look at the first 5 rows of each.

In [ ]:
sql("""
...
""")

In [ ]:
sql("""
...
""")

In [ ]:
sql("""
...
""")

*Which column links `plays` to `users`? Which links `plays` to `songs`? What do you think `referred_by` means?* ...

---
## 1. SELECT, WHERE, ORDER BY, LIMIT

**Example:** build one query a clause at a time. Run each cell and watch the result change. We want to return the customer_id, drink, and price of the three most expensive orders that cost more than $5, sorted from highest price to lowest

In [ ]:
sql("""
SELECT customer_id, drink, price
FROM orders
""")

In [ ]:
sql("""
SELECT customer_id, drink, price
FROM orders
WHERE price > 5
""")

In [ ]:
sql("""
SELECT customer_id, drink, price
FROM orders
WHERE price > 5
ORDER BY price DESC
""")

In [ ]:
sql("""
SELECT customer_id, drink, price
FROM orders
WHERE price > 5
ORDER BY price DESC
LIMIT 3
""")

Same thing in pandas: `orders[orders["price"] > 5].sort_values("price", ascending=False).head(3)[["customer_id", "drink", "price"]]`.


SQL WHERE

In [ ]:
sql("""
SELECT drink, size, price
FROM orders
WHERE drink IN ('Latte', 'Mocha') AND size = 'large'
""")

SQL Date time format: **YYYY-MM-DD HH:MM:SS**

In [ ]:
sql("""
SELECT order_id, order_date
FROM orders
WHERE order_date BETWEEN '2026-10-01' AND '2026-10-07'
""")

**Predict:** `%` is any number of characters, `_` is exactly one. Which drinks match each pattern?

In [ ]:
# Guess:
sql("""
SELECT DISTINCT drink
FROM orders
WHERE drink LIKE '%Latte'
""")

In [ ]:
# Guess:
sql("""
SELECT DISTINCT drink
FROM orders
WHERE drink LIKE '_a%'
""")

**Tinker:** change `'_a%'` to `'%a'`, then `'%e%'`. (In SQLite, `LIKE` ignores upper/lower case. Postgres doesn't.)

**Fix:** this should return Berkeley customers, but it errors. Read the message.

In [ ]:
sql("""
SELECT customer_name, city
FROM customers
WHERE city = 'Berkeley'
ORDER BY customer_name
""")

**Your Turn:** using `songs` and `users`. Seven short ones, in order.

1. Every song released in 2020 or later: title, artist, release year. Newest first.

In [ ]:
sql("""
...
""")

2. Pop or Indie songs (use `IN`) longer than 4 minutes (`duration_sec > 240`).

In [ ]:
sql("""
...
""")

3. Songs released between 2015 and 2019 (`BETWEEN`), oldest first.

In [ ]:
sql("""
...
""")

4. Every song with **Love** anywhere in the title. (Check: should `Lovesick Algorithm` count?)

In [ ]:
sql("""
...
""")

5. Usernames that are **exactly 5 characters** long. (Hint: `_` matches exactly one character.)

In [ ]:
sql("""
...
""")

6. The 5 longest songs: title, artist, duration.

In [ ]:
sql("""
...
""")

7. Premium users in the US or Canada, earliest signup first.

In [ ]:
sql("""
...
""")

---
## 2. Aggregates, GROUP BY, HAVING

**Example:** with no `GROUP BY`, an aggregate squashes the whole table into one row.

In [ ]:
sql("""
SELECT COUNT(*) AS num_orders, SUM(price) AS revenue, AVG(price) AS avg_price
FROM orders
""")

**Predict:** what would this return?

In [ ]:
# Guess:
sql("""
SELECT drink, COUNT(*) AS num_orders, AVG(price) AS avg_price
FROM orders
GROUP BY drink
""")

Same as `orders.groupby("drink")["price"].agg(["count", "mean"])`.

**Example:** group by two columns: one group per unique (drink, size) combo.

In [ ]:
sql("""
SELECT drink, size, COUNT(*) AS num_orders
FROM orders
GROUP BY drink, size
ORDER BY drink, size
""")

Notice the rows where `size` is `None`. `NULL` gets its own group.

**Example:** `HAVING` filters whole groups, after they're counted. Run each cell.

In [ ]:
sql("""
SELECT customer_id, COUNT(*) AS num_orders, SUM(price) AS total_spent
FROM orders
GROUP BY customer_id
""")

In [ ]:
sql("""
SELECT customer_id, COUNT(*) AS num_orders, SUM(price) AS total_spent
FROM orders
GROUP BY customer_id
HAVING SUM(price) > 30
ORDER BY total_spent DESC
""")

**Fix:** this should list customers with more than 4 orders.

In [ ]:
sql("""
SELECT customer_id, COUNT(*) AS num_orders
FROM orders
WHERE COUNT(*) > 4
GROUP BY customer_id
""")

**Your Turn:** aggregates on the music app.

1. How many songs are in each genre? Most songs first. Name the count `num_songs`.

In [ ]:
sql("""
...
""")

2. Average song length per genre **in minutes**. (Hint: `duration_sec / 60.0`. Try `/ 60` too: what goes wrong?)

In [ ]:
sql("""
...
""")

3. Oldest and newest release year in each genre (`MIN`, `MAX`).

In [ ]:
sql("""
...
""")

4. Number of users for every (country, plan) combo, sorted by country then plan.

In [ ]:
sql("""
...
""")

5. Total **minutes** listened per `user_id`, highest first.

In [ ]:
sql("""
...
""")

6. Same as 5, but only users who listened for **more than 2 hours**.

In [ ]:
sql("""
...
""")

7. Genres whose songs average **more than 3.5 minutes**.

In [ ]:
sql("""
...
""")

8. Plays per **month**. `play_date` is text like `'2026-09-14'`, and `SUBSTR(play_date, 1, 7)` keeps the first 7 characters (`'2026-09'`). Group by that.

In [ ]:
sql("""
...
""")

---
## 3. Joins

`orders` only stores `customer_id`. The names live in `customers`. A join puts them back together.

**Example:** `INNER JOIN` keeps only rows that match in **both** tables.

In [ ]:
sql("""
SELECT o.order_id, c.customer_name, o.drink, o.price
FROM orders AS o
INNER JOIN customers AS c ON o.customer_id = c.customer_id
""")

**Predict:** `orders` has 40 rows. How many rows did the inner join return? Why not 40? (Hint: look for a `customer_id` in `orders` that isn't in `customers`.)

In [ ]:
# Guess:
sql("""
SELECT COUNT(*) AS rows_after_inner_join
FROM orders AS o
INNER JOIN customers AS c ON o.customer_id = c.customer_id
""")

In [ ]:
sql("""
SELECT *
FROM orders
WHERE customer_id NOT IN (SELECT customer_id FROM customers)
""")

**Example:** `LEFT JOIN` keeps **every** customer, even ones with no orders. Look for Hana.

In [ ]:
sql("""
SELECT c.customer_name, o.drink, o.price
FROM customers AS c
LEFT JOIN orders AS o ON c.customer_id = o.customer_id
ORDER BY o.price
""")

Same as `pd.merge(customers, orders, on="customer_id", how="left")`.

**Example:** join, then group. Total spent per customer, by **name**. Customers who never ordered show up with 0 orders.

In [ ]:
sql("""
SELECT c.customer_name, COUNT(o.order_id) AS num_orders, SUM(o.price) AS total_spent
FROM customers AS c
LEFT JOIN orders AS o ON c.customer_id = o.customer_id
GROUP BY c.customer_name
ORDER BY total_spent DESC
""")

**Tinker:** change `COUNT(o.order_id)` to `COUNT(*)`. What happens to Hana's count, and why?

**Example:** `FULL OUTER JOIN` keeps unmatched rows from **both** sides: Hana (no orders) **and** the order from customer 9 (no customer). Skip this cell if your SQLite is older than 3.39.

In [ ]:
sql("""
SELECT c.customer_name, o.order_id, o.customer_id
FROM customers AS c
FULL OUTER JOIN orders AS o ON c.customer_id = o.customer_id
WHERE c.customer_id IS NULL OR o.order_id IS NULL
""")

**Predict:** this join forgot its `ON`. 8 customers, 40 orders. How many rows come back?

In [ ]:
# Guess:
sql("""
SELECT COUNT(*) AS num_rows
FROM customers
JOIN orders
""")

Every customer paired with every order. That's a cross join: no error, just a huge wrong table.

**Your Turn:** joins. Some of these need **all three** tables: chain a second `JOIN ... ON` right after the first.

1. Each play with the `username` and song `title` next to it. Show 10 rows.

In [ ]:
sql("""
...
""")

2. Number of plays per **username**, including users with zero plays. Fewest first.

In [ ]:
sql("""
...
""")

3. Songs that have **never** been played. (Hint: `LEFT JOIN`, then keep rows where the play side `IS NULL`.)

In [ ]:
sql("""
...
""")

4. Total minutes listened per **genre**, highest first.

In [ ]:
sql("""
...
""")

5. Top 3 **artists** by number of plays.

In [ ]:
sql("""
...
""")

6. Number of plays from each **country**.

In [ ]:
sql("""
...
""")

7. Which genre do **premium** users play the most? (Three tables, a `WHERE`, a `GROUP BY`.)

In [ ]:
sql("""
...
""")

8. **Challenge:** each user next to the username of whoever referred them. `referred_by` holds a `user_id`, so you join `users` to **itself**. Give the two copies different aliases (`u` and `r`). Keep users nobody referred.

In [ ]:
sql("""
...
""")

---
## 4. CASE, NULLs, Subqueries, CTEs, UNION

**Example:** `CASE` makes a new column with if/else logic. First match wins.

In [ ]:
sql("""
SELECT drink, price,
    CASE
        WHEN price >= 6 THEN 'premium'
        WHEN price >= 5 THEN 'regular'
        ELSE 'budget'
    END AS price_tier
FROM orders
LIMIT 10
""")

**Tinker:** swap the order of the two `WHEN` lines. Why does nothing come out as `'premium'` anymore?

**Example:** `CASE` inside an aggregate counts things conditionally.

In [ ]:
sql("""
SELECT drink,
    COUNT(*) AS num_orders,
    SUM(CASE WHEN size = 'large' THEN 1 ELSE 0 END) AS num_large
FROM orders
GROUP BY drink
""")

**Fix:** this should find orders with no size recorded. It runs, but returns nothing.

In [ ]:
sql("""
SELECT order_id, drink, size
FROM orders
WHERE size = NULL
""")

**Example:** a subquery runs first and hands its answer to the outer query. Run the inner one alone, then the whole thing.

In [ ]:
sql("""
SELECT AVG(price) FROM orders
""")

In [ ]:
sql("""
SELECT order_id, drink, price
FROM orders
WHERE price > (SELECT AVG(price) FROM orders)
ORDER BY price DESC
""")

**Example:** the same "customers who spent over $30" as before, but now with names: a CTE names the intermediate result, then we use it like a table (like saving a pandas step to a variable).

In [ ]:
sql("""
WITH spend AS (
    SELECT customer_id, SUM(price) AS total_spent
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, s.total_spent
FROM spend AS s
INNER JOIN customers AS c ON s.customer_id = c.customer_id
WHERE s.total_spent > 30
ORDER BY s.total_spent DESC
""")

**Predict:** `UNION` removes duplicate rows, `UNION ALL` keeps them. How many rows will each return?

In [ ]:
# Guess: UNION ___ rows, UNION ALL ___ rows
sql("""
SELECT city FROM customers WHERE customer_id <= 3
UNION
SELECT city FROM customers WHERE customer_id >= 6
""")

In [ ]:
sql("""
SELECT city FROM customers WHERE customer_id <= 3
UNION ALL
SELECT city FROM customers WHERE customer_id >= 6
""")

**Your Turn:** CASE, NULLs, subqueries, CTEs, UNION.

1. Label every song `'short'` (under 2.5 min), `'medium'` (2.5 to 4 min), or `'long'` (over 4 min). Show title, duration, and the label.

In [ ]:
sql("""
...
""")

2. Count how many songs got each label from Q1. (Hint: put the whole `CASE ... END` in the `GROUP BY` too, or group by the alias.)

In [ ]:
sql("""
...
""")

3. A play under 30 seconds is a **skip**. For each `user_id`, count total plays and skips (`SUM(CASE ...)`).

In [ ]:
sql("""
...
""")

4. Users nobody referred have `NULL` in `referred_by`. List those users with `IS NULL`.

In [ ]:
sql("""
...
""")

5. Songs **longer than the average** song (subquery).

In [ ]:
sql("""
...
""")

6. Users who **never** played anything (`NOT IN` subquery). Compare with your answer to Joins Q2.

In [ ]:
sql("""
...
""")

7. Songs from the **most-played artist**. (The subquery finds the artist; the outer query lists their songs.)

In [ ]:
sql("""
...
""")

8. With a CTE: each user's total minutes, then the usernames of everyone **above the average** user.

In [ ]:
sql("""
...
""")

9. One column of every **name** in the app: all usernames and all artist names. Add a second column saying `'user'` or `'artist'`. Use `UNION`. Then try `UNION ALL`: why does the row count jump?

In [ ]:
sql("""
...
""")

---
## 5. Saving Results: CREATE TABLE and CREATE VIEW

`sql()` only reads. To change the database, use `conn.execute`.

**Example:** `CREATE TABLE ... AS` saves a query's result as a real table (a snapshot). `CREATE VIEW` saves the **query itself** and reruns it every time you use it.

In [ ]:
conn.execute("DROP TABLE IF EXISTS spend_snapshot")
conn.execute("DROP VIEW IF EXISTS spend_view")

conn.execute("""
CREATE TABLE spend_snapshot AS
SELECT customer_id, SUM(price) AS total_spent FROM orders GROUP BY customer_id
""")
conn.execute("""
CREATE VIEW spend_view AS
SELECT customer_id, SUM(price) AS total_spent FROM orders GROUP BY customer_id
""")
sql("SELECT name, type FROM sqlite_master WHERE name LIKE 'spend%'")

**Predict:** we add a $100 order for customer 1. Which one updates, the snapshot or the view?

In [ ]:
# Guess:
conn.execute("INSERT INTO orders VALUES (41, 1, 'Latte', 'large', 100.0, '2026-10-31')")

print(sql("SELECT * FROM spend_snapshot WHERE customer_id = 1"))
print(sql("SELECT * FROM spend_view WHERE customer_id = 1"))

A **materialized view** (Postgres) is in between: it saves a snapshot like a table, but remembers its query so you can refresh it. SQLite doesn't have them.

Undo the fake order so reruns stay clean:

In [ ]:
conn.execute("DELETE FROM orders WHERE order_id = 41")
conn.commit()

**Your Turn:**

1. Create a view `song_stats`: each song's title, artist, number of plays, and total minutes listened (unplayed songs included, with 0 plays). Then query it for the 5 songs with the most minutes.

In [ ]:
conn.execute("DROP VIEW IF EXISTS song_stats")
conn.execute("""
CREATE VIEW song_stats AS
...
""")
sql("""
...
""")

2. Use your view (no joins needed now): which songs have **0 plays**?

In [ ]:
sql("""
...
""")

3. Save a **snapshot** table `song_stats_snapshot` from the view (`CREATE TABLE ... AS SELECT * FROM song_stats`). Then insert one play of song 27 by user 1 and check song 27 in both. Delete the play afterwards.

In [ ]:
conn.execute("DROP TABLE IF EXISTS song_stats_snapshot")
conn.execute(...)
conn.execute("INSERT INTO plays VALUES (99999, 1, 27, '2026-10-31', 200)")

print(sql(...))
print(sql(...))

conn.execute("DELETE FROM plays WHERE play_id = 99999")
conn.commit()

---
## 6. Challenge Round

These are interview-style: each one takes a few steps. Plan it in plain English first (*which tables? join? group by what? filter before or after grouping?*), then write the SQL. Use a CTE whenever a step needs its own name.

Two new tools you may want:
- `COUNT(DISTINCT column)` counts unique values instead of rows.
- `ROUND(x, 2)` rounds to 2 decimals.

**C1.** Skip rate per genre: the fraction of plays under 30 seconds. Highest skip rate first. (Hint: `AVG(CASE WHEN ... THEN 1.0 ELSE 0 END)`.)

In [ ]:
sql("""
...
""")

**C2.** Which 5 users played the most **different songs**? Show their total plays too. Who replays the same songs the most?

In [ ]:
sql("""
...
""")

**C3.** Do premium users listen more? Average minutes **per user** for each plan. Careful: first total each user, *then* average those totals by plan (CTE). Include users with 0 minutes.

In [ ]:
sql("""
...
""")

**C4.** Users who referred **at least 2** people. Show the referrer's username and how many they referred.

In [ ]:
sql("""
...
""")

**C5.** Did referred users listen more than non-referred users? Use `CASE` to label each user `'referred'` or `'organic'`, then compare average total minutes per user.

In [ ]:
sql("""
...
""")

**C6.** Each song's play count compared with its **genre's** average play count. Show only songs that beat their genre's average. (Hint: CTE 1 = plays per song. CTE 2 = average of CTE 1 per genre. Join them on genre.)

In [ ]:
sql("""
...
""")

**C7.** The **most-played genre in each country**. (Hint: CTE 1 = plays per (country, genre). CTE 2 = the max of that per country. Join them where the count equals the max. Ties can give two rows, and that's fine.)

In [ ]:
sql("""
...
""")

---
## 7. Translate pandas to SQL

You already know pandas. Each cell shows a pandas line on the music data; write the SQL that gives the same table. Run both and compare. (The DataFrames `users`, `songs`, `plays` from the setup cell are still in memory.)

**P1.** Filter and sort.

In [ ]:
songs[songs["genre"] == "Hip-Hop"].sort_values("release_year")[["title", "release_year"]]

In [ ]:
sql("""
...
""")

**P2.** Group and count.

In [ ]:
users.groupby("plan")["user_id"].count()

In [ ]:
sql("""
...
""")

**P3.** Merge, then group.

In [ ]:
pd.merge(plays, songs, on="song_id").groupby("artist")["seconds_listened"].sum().sort_values(ascending=False)

In [ ]:
sql("""
...
""")

**P4.** Left merge and find missing matches.

In [ ]:
m = pd.merge(users, plays, on="user_id", how="left")
m[m["play_id"].isna()][["username"]]

In [ ]:
sql("""
...
""")

---
## Done: Push Your Work

Run these in your terminal (not in this notebook). Don't commit `week7.db`; the notebook rebuilds it.

```
git add Week7.ipynb
git commit -m "week 7 notebook"
git push origin YOUR-BRANCH-NAME
```